In [ ]:
# prompt: a notebook showing how to implement U-net on a small dataset with pytorch

# Install necessary libraries
!pip install torch torchvision matplotlib albumentations

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

# Set a seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# --- 1. Prepare a Small Sample Dataset ---
# Since we don't have a real dataset, let's create a dummy one.
# We'll generate random "images" and "masks".
# In a real scenario, you would load actual image and mask files.

DATA_DIR = './unet_sample_data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
MASKS_DIR = os.path.join(DATA_DIR, 'masks')

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR, exist_ok=True)

NUM_SAMPLES = 20
IMAGE_SIZE = 128

for i in range(NUM_SAMPLES):
    # Create a random image (grayscale)
    img = np.random.randint(0, 256, (IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    img_path = os.path.join(IMAGES_DIR, f'image_{i:03d}.png')
    Image.fromarray(img).save(img_path)

    # Create a random mask (binary, e.g., background=0, foreground=1)
    mask = np.random.randint(0, 2, (IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8) * 255 # Use 255 for visualization
    mask_path = os.path.join(MASKS_DIR, f'mask_{i:03d}.png')
    Image.fromarray(mask, mode='L').save(mask_path) # Save as grayscale mask


print(f"Created {NUM_SAMPLES} dummy samples in {DATA_DIR}")

# --- 2. Define the Custom Dataset Class ---
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))
        assert len(self.images) == len(self.masks), "Number of images and masks must be equal."

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = np.array(Image.open(img_path).convert("L")) # Convert to grayscale (1 channel)
        mask = np.array(Image.open(mask_path).convert("L")) # Convert to grayscale (1 channel)
        mask[mask > 0] = 1 # Ensure mask is binary (0 or 1)

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        # Add channel dimension to mask if needed (for binary segmentation)
        mask = mask.unsqueeze(0) if mask.ndim == 2 else mask

        return image, mask.float() # Ensure mask is float for BCEWithLogitsLoss


# --- 3. Define Transformations ---
train_transform = A.Compose([
    A.Resize(width=IMAGE_SIZE, height=IMAGE_SIZE),
    A.Rotate(limit=35, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Normalize(
        mean=[0.5], # Adjust mean and std for 1-channel grayscale
        std=[0.5],
        max_pixel_value=255.0,
    ),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(width=IMAGE_SIZE, height=IMAGE_SIZE),
    A.Normalize(
        mean=[0.5], # Adjust mean and std for 1-channel grayscale
        std=[0.5],
        max_pixel_value=255.0,
    ),
    ToTensorV2(),
])

# --- 4. Split Data and Create DataLoaders ---
# For a small dataset, a simple split is fine.
# In a real project, use train_test_split from sklearn.
train_size = int(0.8 * NUM_SAMPLES)
val_size = NUM_SAMPLES - train_size

all_image_files = sorted(os.listdir(IMAGES_DIR))
all_mask_files = sorted(os.listdir(MASKS_DIR))

train_images = all_image_files[:train_size]
val_images = all_image_files[train_size:]

train_masks = all_mask_files[:train_size]
val_masks = all_mask_files[train_size:]

# To use the same Dataset class, we'll create temporary subdirectories
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, 'train_images')
TRAIN_MASKS_DIR = os.path.join(DATA_DIR, 'train_masks')
VAL_IMAGES_DIR = os.path.join(DATA_DIR, 'val_images')
VAL_MASKS_DIR = os.path.join(DATA_DIR, 'val_masks')

os.makedirs(TRAIN_IMAGES_DIR, exist_ok=True)
os.makedirs(TRAIN_MASKS_DIR, exist_ok=True)
os.makedirs(VAL_IMAGES_DIR, exist_ok=True)
os.makedirs(VAL_MASKS_DIR, exist_ok=True)

import shutil

for img_file, mask_file in zip(train_images, train_masks):
    shutil.copy(os.path.join(IMAGES_DIR, img_file), os.path.join(TRAIN_IMAGES_DIR, img_file))
    shutil.copy(os.path.join(MASKS_DIR, mask_file), os.path.join(TRAIN_MASKS_DIR, mask_file))

for img_file, mask_file in zip(val_images, val_masks):
    shutil.copy(os.path.join(IMAGES_DIR, img_file), os.path.join(VAL_IMAGES_DIR, img_file))
    shutil.copy(os.path.join(MASKS_DIR, mask_file), os.path.join(VAL_MASKS_DIR, mask_file))


BATCH_SIZE = 4
NUM_WORKERS = 2 # Set to 0 on Windows if encountering issues
PIN_MEMORY = True # Set to False on Windows if encountering issues

train_dataset = SegmentationDataset(image_dir=TRAIN_IMAGES_DIR, mask_dir=TRAIN_MASKS_DIR, transform=train_transform)
val_dataset = SegmentationDataset(image_dir=VAL_IMAGES_DIR, mask_dir=VAL_MASKS_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


# --- 5. Define the U-Net Architecture ---
# Helper function for a convolutional block
def double_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

# The U-Net model
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1): # in_channels=1 for grayscale, out_channels=1 for binary segmentation
        super(UNet, self).__init__()

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Contracting Path (Encoder)
        self.down_conv1 = double_conv(in_channels, 64)
        self.down_conv2 = double_conv(64, 128)
        self.down_conv3 = double_conv(128, 256)
        self.down_conv4 = double_conv(256, 512)
        self.down_conv5 = double_conv(512, 1024)

        # Expanding Path (Decoder)
        self.up_trans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv1 = double_conv(1024, 512) # Input is concatenation of up_trans1 output and down_conv4 output

        self.up_trans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_conv2 = double_conv(512, 256)

        self.up_trans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv3 = double_conv(256, 128)

        self.up_trans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv4 = double_conv(128, 64)

        # Output layer
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Contracting Path
        x1 = self.down_conv1(x)
        x2 = self.maxpool(x1)

        x3 = self.down_conv2(x2)
        x4 = self.maxpool(x3)

        x5 = self.down_conv3(x4)
        x6 = self.maxpool(x5)

        x7 = self.down_conv4(x6)
        x8 = self.maxpool(x7)

        x9 = self.down_conv5(x8)

        # Expanding Path
        x = self.up_trans1(x9)
        # Concatenate requires matching spatial dimensions.
        # If shapes don't match exactly due to padding/stride, you might need
        # to crop the contracting path output. For simplicity here, assuming
        # perfect shape matching with 2x stride and 2x transpose stride.
        x = torch.cat([x, x7], dim=1) # Concatenate along the channel dimension
        x = self.up_conv1(x)

        x = self.up_trans2(x)
        x = torch.cat([x, x5], dim=1)
        x = self.up_conv2(x)

        x = self.up_trans3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.up_conv3(x)

        x = self.up_trans4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up_conv4(x)

        output = self.out_conv(x)

        return output

# --- 6. Setup Model, Loss Function, and Optimizer ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet(in_channels=1, out_channels=1).to(DEVICE) # Binary segmentation output channel is 1
LEARNING_RATE = 1e-4
# Use BCEWithLogitsLoss for binary segmentation (combines Sigmoid and BCELoss)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 7. Training Loop ---
NUM_EPOCHS = 5 # Reduced for a quick example

def train_fn(loader, model, optimizer, loss_fn, scaler):
    model.train()
    loop = tqdm(loader)
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(DEVICE)
        targets = targets.to(DEVICE)

        # Forward
        with torch.cuda.amp.autocast():
            predictions = model(data)
            loss = loss_fn(predictions, targets)

        # Backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update tqdm loop
        loop.set_postfix(loss=loss.item())

def check_accuracy(loader, model, device="cuda"):
    model.eval()
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            # Predictions are logits, apply sigmoid to get probabilities
            preds = torch.sigmoid(model(x))
            # Threshold probabilities to get binary mask
            preds = (preds > 0.5).float()

            # Accuracy
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)

            # Dice Score (F1 Score)
            # Intersection: preds * y
            # Sum of areas: preds.sum() + y.sum()
            # Dice = 2 * (Intersection) / (Sum of areas)
            dice_score += (2 * (preds * y).sum()) / ((preds + y).sum() + 1e-8) # Add small epsilon to avoid division by zero

    print(f"Got {num_correct}/{num_pixels} with accuracy {num_correct/num_pixels:.4f}")
    print(f"Dice score: {dice_score/len(loader):.4f}")
    model.train()


# For mixed precision training
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()

# Import tqdm for progress bar
from tqdm import tqdm

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    train_fn(train_loader, model, optimizer, loss_fn, scaler)

    # Check accuracy on validation set
    check_accuracy(val_loader, model, device=DEVICE)


print("Training finished.")


# --- 8. Visualize Predictions (Optional) ---
def predict_and_visualize(loader, model, device="cuda", num_samples_to_show=3):
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= num_samples_to_show:
                break
            x = x.to(device)
            y = y.to(device) # Ground truth mask

            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float() # Predicted binary mask

            # Move tensors back to CPU for visualization
            x_cpu = x.cpu().squeeze(1) # Remove channel dim for grayscale
            y_cpu = y.cpu().squeeze(1)
            preds_cpu = preds.cpu().squeeze(1)

            for j in range(x.size(0)): # Iterate through batch
                plt.figure(figsize=(12, 4))

                plt.subplot(1, 3, 1)
                plt.imshow(x_cpu[j], cmap='gray')
                plt.title("Input Image")
                plt.axis('off')

                plt.subplot(1, 3, 2)
                plt.imshow(y_cpu[j], cmap='gray')
                plt.title("Ground Truth Mask")
                plt.axis('off')

                plt.subplot(1, 3, 3)
                plt.imshow(preds_cpu[j], cmap='gray')
                plt.title("Predicted Mask")
                plt.axis('off')

                plt.show()
    model.train() # Set model back to training mode

print("\nVisualizing some predictions:")
predict_and_visualize(val_loader, model, device=DEVICE, num_samples_to_show=3)

# Clean up the dummy data directories
# shutil.rmtree(DATA_DIR)
# print(f"Cleaned up dummy data directory: {DATA_DIR}")
